Import

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_validate
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, roc_curve

import sklearn
print('scikit-learn:', sklearn.__version__)

DATA = 'train.csv'


load & Cek Data

In [ ]:
df = pd.read_csv(DATA)
print('Shape  :', df.shape)
print('Nulls  :', df.isnull().sum().sum())
print('Target :\n', df['target'].value_counts())
df.head(3)


feature Engineering

In [ ]:
def add_features(df):
    df = df.copy()
    df['loan_to_income']    = df['num_3'] / (df['num_2'] + 1)
    df['credit_util_ratio'] = df['num_3'] / (df['num_4'] + 1)
    df['interest_x_months'] = df['num_7'] * df['num_8']
    df['debt_burden']       = df['num_9'] * df['num_3']
    df['age_x_credit']      = df['num_1'] * df['num_4']
    df['income_per_month']  = df['num_2'] / (df['num_5'] + 1)
    return df

df = add_features(df)
print('Total fitur setelah FE:', df.shape[1] - 2)


Split Train / Validation

In [ ]:
CAT_COLS = [f'cat_{i}' for i in range(1, 8)]
NUM_COLS = [c for c in df.columns if c not in CAT_COLS + ['id', 'target']]

X = df.drop(columns=['id', 'target'])
y = df['target']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y
)
print('Train:', X_train.shape, '| Val:', X_val.shape)


Preprocessor

In [ ]:
# Preprocessor dasar: StandardScaler + OrdinalEncoder
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), NUM_COLS),
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), CAT_COLS)
])

# Preprocessor dengan PCA (untuk Logistic Regression)
preprocessor_pca = Pipeline([
    ('ct', ColumnTransformer([
        ('num', StandardScaler(), NUM_COLS),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), CAT_COLS)
    ])),
    ('pca', PCA(n_components=0.95))
])


Model 1 — HistGradientBoostingClassifier + Tuning

In [ ]:
pipe_hgb = Pipeline([
    ('pre', preprocessor),
    ('clf', HistGradientBoostingClassifier(class_weight='balanced'))
])

param_hgb = {
    'clf__max_iter'          : [200, 300, 500],
    'clf__learning_rate'     : [0.05, 0.1, 0.2],
    'clf__max_depth'         : [3, 5, 7],
    'clf__min_samples_leaf'  : [20, 50, 100],
    'clf__l2_regularization' : [0.0, 0.1, 1.0],
}

search_hgb = RandomizedSearchCV(
    pipe_hgb, param_hgb,
    n_iter=30, cv=5, scoring='roc_auc',
    n_jobs=-1, verbose=1
)
search_hgb.fit(X_train, y_train)
best_hgb = search_hgb.best_estimator_

auc_hgb = roc_auc_score(y_val, best_hgb.predict_proba(X_val)[:, 1])
f1_hgb  = f1_score(y_val, best_hgb.predict(X_val))
acc_hgb = accuracy_score(y_val, best_hgb.predict(X_val))

print('=== HistGradientBoosting ===')
print(f'Best params : {search_hgb.best_params_}')
print(f'AUC-ROC     : {auc_hgb:.4f}')
print(f'F1-Score    : {f1_hgb:.4f}')
print(f'Accuracy    : {acc_hgb:.4f}')


Cross Validation — HistGradientBoosting

In [ ]:
cv_hgb = cross_validate(
    best_hgb, X, y, cv=5,
    scoring=['roc_auc', 'f1', 'accuracy'],
    n_jobs=-1
)

print('=== CV HistGradientBoosting (5-fold) ===')
print(f'AUC-ROC  : {cv_hgb["test_roc_auc"].mean():.4f} (+/- {cv_hgb["test_roc_auc"].std():.4f})')
print(f'F1-Score : {cv_hgb["test_f1"].mean():.4f} (+/- {cv_hgb["test_f1"].std():.4f})')
print(f'Accuracy : {cv_hgb["test_accuracy"].mean():.4f} (+/- {cv_hgb["test_accuracy"].std():.4f})')


model 2 — Logistic Regression + Tuning

In [ ]:
pipe_lr = Pipeline([
    ('pre', preprocessor_pca),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000))
])

param_lr = {
    'clf__C'      : [0.001, 0.01, 0.1, 1, 10, 100],
    'clf__penalty': ['l1', 'l2'],
    'clf__solver' : ['liblinear', 'saga'],
}

search_lr = RandomizedSearchCV(
    pipe_lr, param_lr,
    n_iter=20, cv=5, scoring='roc_auc',
    n_jobs=-1, verbose=1
)
search_lr.fit(X_train, y_train)
best_lr = search_lr.best_estimator_

auc_lr = roc_auc_score(y_val, best_lr.predict_proba(X_val)[:, 1])
f1_lr  = f1_score(y_val, best_lr.predict(X_val))
acc_lr = accuracy_score(y_val, best_lr.predict(X_val))

print('=== Logistic Regression ===')
print(f'Best params : {search_lr.best_params_}')
print(f'AUC-ROC     : {auc_lr:.4f}')
print(f'F1-Score    : {f1_lr:.4f}')
print(f'Accuracy    : {acc_lr:.4f}')


Cross Validation — Logistic Regression

In [ ]:
cv_lr = cross_validate(
    best_lr, X, y, cv=5,
    scoring=['roc_auc', 'f1', 'accuracy'],
    n_jobs=-1
)

print('=== CV Logistic Regression (5-fold) ===')
print(f'AUC-ROC  : {cv_lr["test_roc_auc"].mean():.4f} (+/- {cv_lr["test_roc_auc"].std():.4f})')
print(f'F1-Score : {cv_lr["test_f1"].mean():.4f} (+/- {cv_lr["test_f1"].std():.4f})')
print(f'Accuracy : {cv_lr["test_accuracy"].mean():.4f} (+/- {cv_lr["test_accuracy"].std():.4f})')


model 3 — Random Forest + Tuning

In [ ]:
pipe_rf = Pipeline([
    ('pre', preprocessor),
    ('clf', RandomForestClassifier(class_weight='balanced', n_jobs=-1))
])

param_rf = {
    'clf__n_estimators'    : [200, 300, 500],
    'clf__max_depth'       : [5, 10, 15, None],
    'clf__min_samples_leaf': [10, 20, 50],
    'clf__max_features'    : ['sqrt', 'log2', 0.5],
}

search_rf = RandomizedSearchCV(
    pipe_rf, param_rf,
    n_iter=25, cv=5, scoring='roc_auc',
    n_jobs=-1, verbose=1
)
search_rf.fit(X_train, y_train)
best_rf = search_rf.best_estimator_

auc_rf = roc_auc_score(y_val, best_rf.predict_proba(X_val)[:, 1])
f1_rf  = f1_score(y_val, best_rf.predict(X_val))
acc_rf = accuracy_score(y_val, best_rf.predict(X_val))

print('=== Random Forest ===')
print(f'Best params : {search_rf.best_params_}')
print(f'AUC-ROC     : {auc_rf:.4f}')
print(f'F1-Score    : {f1_rf:.4f}')
print(f'Accuracy    : {acc_rf:.4f}')


Cross Validation — Random Forest

In [ ]:
cv_rf = cross_validate(
    best_rf, X, y, cv=5,
    scoring=['roc_auc', 'f1', 'accuracy'],
    n_jobs=-1
)

print('=== CV Random Forest (5-fold) ===')
print(f'AUC-ROC  : {cv_rf["test_roc_auc"].mean():.4f} (+/- {cv_rf["test_roc_auc"].std():.4f})')
print(f'F1-Score : {cv_rf["test_f1"].mean():.4f} (+/- {cv_rf["test_f1"].std():.4f})')
print(f'Accuracy : {cv_rf["test_accuracy"].mean():.4f} (+/- {cv_rf["test_accuracy"].std():.4f})')


Ringkasan Hasil Model

In [ ]:
results = pd.DataFrame([
    {'Model': 'HistGradientBoosting', 'AUC-ROC': auc_hgb, 'F1-Score': f1_hgb, 'Accuracy': acc_hgb},
    {'Model': 'LogisticRegression',   'AUC-ROC': auc_lr,  'F1-Score': f1_lr,  'Accuracy': acc_lr},
    {'Model': 'RandomForest',         'AUC-ROC': auc_rf,  'F1-Score': f1_rf,  'Accuracy': acc_rf},
]).sort_values('AUC-ROC', ascending=False).reset_index(drop=True)

print(results.to_string(index=False))
best_model_name = results.iloc[0]['Model']
print(f'\nBest Model: {best_model_name}')


ROC Curve Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

models_info = [
    (best_hgb, 'HistGradientBoosting', '#E53935'),
    (best_lr,  'LogisticRegression',   '#1E88E5'),
    (best_rf,  'RandomForest',         '#43A047'),
]

for model, name, color in models_info:
    y_prob = model.predict_proba(X_val)[:, 1]
    fpr, tpr, _ = roc_curve(y_val, y_prob)
    auc = roc_auc_score(y_val, y_prob)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC = {auc:.4f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1.2, label='Random Classifier')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve Comparison — Loan Default Prediction', fontsize=13)
ax.legend(loc='lower right', fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('roc_comparison.png', dpi=150)
plt.show()
print('Plot saved: roc_comparison.png')


Prediksi Test Set

In [ ]:
import os

TEST_PATH = 'test.csv'

if os.path.exists(TEST_PATH):
    df_test = pd.read_csv(TEST_PATH)
    df_test = add_features(df_test)

    best_models = {
        'HistGradientBoosting': best_hgb,
        'LogisticRegression'  : best_lr,
        'RandomForest'        : best_rf,
    }
    chosen_model = best_models[best_model_name]

    X_test_final = df_test.drop(columns=['id'])
    prob = chosen_model.predict_proba(X_test_final)[:, 1]
    pred = (prob >= 0.5).astype(int)

    submission = pd.DataFrame({
        'id'         : df_test['id'],
        'prediction' : pred,
        'probability': prob.round(6),
    })
    submission.to_csv('nama_team_predictions.csv', index=False)
    print('Submission saved: nama_team_predictions.csv')
    print(submission.head())
else:
    print(f'File {TEST_PATH} belum ada. Letakkan test.csv lalu jalankan cell ini.')
